In [8]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Flatten,
    Dense,
    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

In [9]:
# ==========================================
# DATASET PATHS
# ==========================================

BASE_DIR = "smartvision_dataset"

TRAIN_DIR = f"{BASE_DIR}/classification/train"
VAL_DIR   = f"{BASE_DIR}/classification/val"
TEST_DIR  = f"{BASE_DIR}/classification/test"

In [10]:
# ==========================================
# CREATE FOLDERS
# ==========================================

os.makedirs("models", exist_ok=True)
os.makedirs("outputs/plots", exist_ok=True)

In [11]:
# ==========================================
# IMAGE SETTINGS
# ==========================================

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [12]:
# ==========================================
# DATA GENERATORS
# SIMPLE NORMALIZATION
# ==========================================

train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=10,

    zoom_range=0.1,

    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(

    rescale=1./255
)

In [13]:
# ==========================================
# LOAD DATA
# ==========================================

train_gen = train_datagen.flow_from_directory(

    TRAIN_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    class_mode='categorical',

    shuffle=True,

    seed=42
)

val_gen = val_test_datagen.flow_from_directory(

    VAL_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    class_mode='categorical',

    shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(

    TEST_DIR,

    target_size=IMG_SIZE,

    batch_size=BATCH_SIZE,

    class_mode='categorical',

    shuffle=False
)


Found 1820 images belonging to 26 classes.
Found 390 images belonging to 26 classes.
Found 390 images belonging to 26 classes.


In [14]:
# ==========================================
# LOAD VGG16
# ==========================================

base_model = VGG16(

    weights='imagenet',

    include_top=False,

    input_shape=(224, 224, 3)
)


In [15]:
# ==========================================
# FREEZE ALL LAYERS
# ==========================================

for layer in base_model.layers:
    layer.trainable = False


In [16]:
# ==========================================
# BUILD MODEL
# ==========================================

model = Sequential([

    base_model,

    Flatten(),

    Dense(512, activation='relu'),

    Dropout(0.5),

    Dense(train_gen.num_classes, activation='softmax')
])


In [17]:
# ==========================================
# COMPILE MODEL
# ==========================================

model.compile(

    optimizer=Adam(learning_rate=0.0001),

    loss='categorical_crossentropy',

    metrics=['accuracy']
)


In [18]:
# ==========================================
# MODEL SUMMARY
# ==========================================

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg16 (Functional)              │ (None, 7, 7, 512)      │    14,714,688 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 512)            │    12,845,568 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 26)             │        13,338 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,573,594 (105.18 MB)

 Trainable params: 12,858,906 (49.05 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [19]:
# ==========================================
# CALLBACKS
# ==========================================

callbacks = [

    EarlyStopping(

        monitor='val_loss',

        patience=5,

        restore_best_weights=True
    ),

    ReduceLROnPlateau(

        monitor='val_loss',

        factor=0.2,

        patience=2,

        verbose=1
    ),

    ModelCheckpoint(

        "models/vgg16_best.h5",

        monitor='val_accuracy',

        save_best_only=True,

        verbose=1
    )
]

In [20]:
# ==========================================
# TRAIN MODEL
# ==========================================

EPOCHS = 15

history = model.fit(

    train_gen,

    validation_data=val_gen,

    epochs=EPOCHS,

    callbacks=callbacks
)

Epoch 1/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1048 - loss: 3.3649
Epoch 1: val_accuracy improved from None to 0.17949, saving model to models/vgg16_best.h5



Epoch 1: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 269s 5s/step - accuracy: 0.1527 - loss: 3.1136 - val_accuracy: 0.1795 - val_loss: 2.9290 - learning_rate: 1.0000e-04
Epoch 2/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.3129 - loss: 2.5166
Epoch 2: val_accuracy improved from 0.17949 to 0.26410, saving model to models/vgg16_best.h5



Epoch 2: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 253s 4s/step - accuracy: 0.3505 - loss: 2.4008 - val_accuracy: 0.2641 - val_loss: 2.6817 - learning_rate: 1.0000e-04
Epoch 3/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4732 - loss: 2.0199
Epoch 3: val_accuracy improved from 0.26410 to 0.26923, saving model to models/vgg16_best.h5



Epoch 3: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 242s 4s/step - accuracy: 0.4725 - loss: 1.9611 - val_accuracy: 0.2692 - val_loss: 2.6028 - learning_rate: 1.0000e-04
Epoch 4/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.5475 - loss: 1.7368
Epoch 4: val_accuracy improved from 0.26923 to 0.28462, saving model to models/vgg16_best.h5



Epoch 4: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 242s 4s/step - accuracy: 0.5533 - loss: 1.6776 - val_accuracy: 0.2846 - val_loss: 2.5273 - learning_rate: 1.0000e-04
Epoch 5/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.6129 - loss: 1.5020
Epoch 5: val_accuracy improved from 0.28462 to 0.28974, saving model to models/vgg16_best.h5


...


Epoch 6: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 232s 4s/step - accuracy: 0.6758 - loss: 1.3045 - val_accuracy: 0.3128 - val_loss: 2.4658 - learning_rate: 1.0000e-04
Epoch 7/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.6814 - loss: 1.2378
Epoch 7: val_accuracy improved from 0.31282 to 0.31795, saving model to models/vgg16_best.h5



Epoch 7: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 243s 4s/step - accuracy: 0.6907 - loss: 1.1947 - val_accuracy: 0.3179 - val_loss: 2.5333 - learning_rate: 1.0000e-04
Epoch 8/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7335 - loss: 1.0759
Epoch 8: ReduceLROnPlateau reducing learning rate to 1.9999999494757503e-05.

Epoch 8: val_accuracy did not improve from 0.31795
57/57 ━━━━━━━━━━━━━━━━━━━━ 230s 4s/step - accuracy: 0.7335 - loss: 1.0772 - val_accuracy: 0.2974 - val_loss: 2.5737 - learning_rate: 1.0000e-04
Epoch 9/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7624 - loss: 0.9662
Epoch 9: val_accuracy improved from 0.31795 to 0.32051, saving model to models/vgg16_best.h5



Epoch 9: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 237s 4s/step - accuracy: 0.7604 - loss: 0.9574 - val_accuracy: 0.3205 - val_loss: 2.4824 - learning_rate: 2.0000e-05
Epoch 10/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7792 - loss: 0.8932
Epoch 10: ReduceLROnPlateau reducing learning rate to 3.999999898951501e-06.

Epoch 10: val_accuracy improved from 0.32051 to 0.32564, saving model to models/vgg16_best.h5



Epoch 10: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 238s 4s/step - accuracy: 0.7879 - loss: 0.8769 - val_accuracy: 0.3256 - val_loss: 2.4952 - learning_rate: 2.0000e-05
Epoch 11/15
57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7805 - loss: 0.8671
Epoch 11: val_accuracy improved from 0.32564 to 0.33333, saving model to models/vgg16_best.h5



Epoch 11: finished saving model to models/vgg16_best.h5
57/57 ━━━━━━━━━━━━━━━━━━━━ 231s 4s/step - accuracy: 0.7819 - loss: 0.8632 - val_accuracy: 0.3333 - val_loss: 2.4908 - learning_rate: 4.0000e-06


In [21]:
# ==========================================
# SAVE FINAL MODEL
# ==========================================

model.save("models/vgg16_final.h5")

print("\n✅ Final model saved successfully")


✅ Final model saved successfully


In [22]:
# ==========================================
# EVALUATE MODEL
# ==========================================

test_loss, test_acc = model.evaluate(test_gen)

print("\n==============================")
print("TEST RESULTS")
print("==============================")

print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test Loss     : {test_loss:.4f}")

13/13 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step - accuracy: 0.2436 - loss: 2.6029

TEST RESULTS
Test Accuracy : 0.2436
Test Loss     : 2.6029


In [23]:
# ==========================================
# PREDICTIONS
# ==========================================

test_gen.reset()

pred_probs = model.predict(test_gen)

y_pred = np.argmax(pred_probs, axis=1)

y_true = test_gen.classes

class_names = list(test_gen.class_indices.keys())

13/13 ━━━━━━━━━━━━━━━━━━━━ 41s 3s/step


In [24]:
# ==========================================
# CLASSIFICATION REPORT
# ==========================================

print("\n==============================")
print("CLASSIFICATION REPORT")
print("==============================\n")

print(classification_report(

    y_true,

    y_pred,

    target_names=class_names,

    zero_division=0
))


CLASSIFICATION REPORT

               precision    recall  f1-score   support

     airplane       0.92      0.73      0.81        15
          bed       0.42      0.53      0.47        15
        bench       0.33      0.47      0.39        15
      bicycle       0.03      0.07      0.05        15
         bird       0.00      0.00      0.00        15
       bottle       0.00      0.00      0.00        15
         bowl       0.50      0.33      0.40        15
          bus       0.31      0.27      0.29        15
         cake       0.17      0.07      0.10        15
          car       0.00      0.00      0.00        15
          cat       0.21      0.53      0.30        15
        chair       0.00      0.00      0.00        15
        couch       0.20      0.07      0.10        15
          cow       0.50      0.07      0.12        15
          cup       0.00      0.00      0.00        15
          dog       0.00      0.00      0.00        15
     elephant       0.69      0.60      

In [25]:
# ==========================================
# METRICS
# ==========================================

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(
    y_true,
    y_pred,
    average='weighted',
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    average='weighted',
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    average='weighted',
    zero_division=0
)

print("\n==============================")
print("EVALUATION METRICS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")


EVALUATION METRICS
Accuracy  : 0.2436
Precision : 0.2525
Recall    : 0.2436
F1 Score  : 0.2311
